# Data Workflow: Wine Quality

**Author:** Vivek Kumar

**Dataset:** Wine Quality (Red + White) — chemical properties and quality scores, from the UCI Machine Learning Repository ([dataset link](https://archive.ics.uci.edu/dataset/186/wine+quality))

**Project description:** This project builds a complete, reproducible data workflow on the UCI Wine Quality dataset, combining the red and white wine samples into a single dataset distinguished by wine type. It loads, cleans, transforms, explores, and visualizes the physicochemical measurements to understand how they relate to expert quality scores. The goal is to establish a clean, well-documented analytical foundation that future machine learning and AI projects can build on.

## 1. Setup

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Load Data

The dataset ships as two semicolon-separated CSV files (red and white). We load each, tag it with a `wine_type` column, and concatenate the two into a single DataFrame so both varieties can be analyzed together.

In [17]:
# Files use ';' as the separator
red = pd.read_csv('data/winequality-red.csv', sep=';')
white = pd.read_csv('data/winequality-white.csv', sep=';')

# Tag each row with its wine type before combining
red['wine_type'] = 'red'
white['wine_type'] = 'white'

# Combine into a single DataFrame
df = pd.concat([red, white], ignore_index=True)

print(f'Red:   {red.shape[0]} rows')
print(f'White: {white.shape[0]} rows')
print(f'Combined: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

Red:   1599 rows
White: 4898 rows
Combined: 6497 rows, 13 columns


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


## 3. Clean Data

We define reusable cleaning functions, each with a docstring describing its purpose, and then apply them to the combined dataset. Writing the cleaning steps as functions keeps the workflow reproducible: the same transformations can be re-run on new data or re-used in later projects.

In [18]:
def clean_column_names(df):
    """Standardize DataFrame column names for consistent, code-friendly access.

    Each column name is lowercased, stripped of leading/trailing whitespace, and
    has internal spaces replaced with underscores (e.g. 'fixed acidity' ->
    'fixed_acidity'). This prevents subtle bugs caused by inconsistent naming and
    enables attribute-style column access.

    Args:
        df (pd.DataFrame): The DataFrame whose columns should be standardized.

    Returns:
        pd.DataFrame: A copy of the DataFrame with cleaned column names.
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
    )
    return df

In [19]:
def drop_duplicate_rows(df):
    """Remove exact duplicate rows from the DataFrame.

    The Wine Quality dataset contains many identical records (same chemical
    measurements and quality score), which can bias summary statistics and
    visualizations. This function drops fully duplicated rows, resets the index,
    and reports how many rows were removed.

    Args:
        df (pd.DataFrame): The DataFrame to de-duplicate.

    Returns:
        pd.DataFrame: A copy of the DataFrame with duplicate rows removed.
    """
    df = df.copy()
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    removed = before - len(df)
    print(f'Removed {removed} duplicate rows ({before} -> {len(df)}).')
    return df

In [20]:
def report_missing_values(df):
    """Summarize missing values per column.

    Returns a tidy summary of how many values are missing in each column and the
    corresponding percentage, sorted from most to least missing. This is a
    diagnostic step that documents data completeness before analysis; the Wine
    Quality dataset is expected to be complete, and this function makes that
    assumption explicit and verifiable.

    Args:
        df (pd.DataFrame): The DataFrame to inspect.

    Returns:
        pd.DataFrame: A summary with 'missing_count' and 'missing_pct' columns.
    """
    missing = df.isnull().sum()
    summary = pd.DataFrame({
        'missing_count': missing,
        'missing_pct': (missing / len(df) * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

### Apply the cleaning functions

In [21]:
# Apply cleaning functions in sequence
df = clean_column_names(df)
df = drop_duplicate_rows(df)

# Diagnostic: confirm there are no missing values
report_missing_values(df)

Removed 1177 duplicate rows (6497 -> 5320).


,missing_count,missing_pct
fixed_acidity,0,0.0
volatile_acidity,0,0.0
citric_acid,0,0.0
residual_sugar,0,0.0
chlorides,0,0.0
free_sulfur_dioxide,0,0.0
total_sulfur_dioxide,0,0.0
density,0,0.0
ph,0,0.0
sulphates,0,0.0


In [22]:
# Verify the result of cleaning
print(f'Cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print('Columns:', list(df.columns))
df.head()

Cleaned dataset: 5320 rows, 13 columns
Columns: ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol', 'quality', 'wine_type']


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5,red


## 4. Exploratory Data Analysis (EDA)

In [23]:
def summarize_dataset(df):
    """Print an overview of the dataset and return its summary statistics.

    Reports the shape, the count of numeric vs. non-numeric columns, and the
    per-column data types, then returns transposed descriptive statistics
    (count, mean, std, min, quartiles, max) for all numeric columns. This gives
    a quick sense of scale, spread, and potential outliers for each feature.

    Args:
        df (pd.DataFrame): The DataFrame to summarize.

    Returns:
        pd.DataFrame: Transposed `describe()` output for the numeric columns.
    """
    print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
    numeric_cols = df.select_dtypes(include=np.number).columns
    print(f'Numeric columns: {len(numeric_cols)} | Non-numeric: {df.shape[1] - len(numeric_cols)}')
    print('\nData types:')
    print(df.dtypes)
    return df.describe().T.round(3)

In [24]:
def group_summary(df, group_col='wine_type', value_col='quality'):
    """Summarize a value column across the levels of a grouping column.

    For each group (e.g. red vs. white wine) this reports the count, mean,
    standard deviation, minimum, and maximum of `value_col`. It is a grouped /
    aggregated view that highlights how a metric such as quality differs across
    categories.

    Args:
        df (pd.DataFrame): The DataFrame to analyze.
        group_col (str): Column to group by. Defaults to 'wine_type'.
        value_col (str): Numeric column to aggregate. Defaults to 'quality'.

    Returns:
        pd.DataFrame: One row per group with aggregated statistics.
    """
    summary = (
        df.groupby(group_col)[value_col]
        .agg(['count', 'mean', 'std', 'min', 'max'])
        .round(3)
    )
    return summary

In [25]:
def correlations_with(df, target='quality', n=None):
    """Rank numeric features by their correlation with a target column.

    Computes the Pearson correlation between every numeric feature and `target`,
    excluding the target's self-correlation, and returns them sorted from most
    positive to most negative. This is a correlation check that surfaces which
    chemical properties move most strongly with the quality score.

    Args:
        df (pd.DataFrame): The DataFrame to analyze.
        target (str): The column to correlate other features against.
        n (int, optional): If given, return only the top `n` features by
            absolute correlation. Defaults to None (return all).

    Returns:
        pd.Series: Correlations with `target`, sorted descending.
    """
    corr = df.corr(numeric_only=True)[target].drop(labels=[target])
    corr = corr.sort_values(ascending=False)
    if n is not None:
        corr = corr.reindex(corr.abs().sort_values(ascending=False).index).head(n)
    return corr.round(3)

### Run the EDA functions

In [26]:
# Summary statistics for all numeric features
summarize_dataset(df)

Shape: 5320 rows x 13 columns
Numeric columns: 12 | Non-numeric: 1

Data types:
fixed_acidity           float64
volatile_acidity        float64
citric_acid             float64
residual_sugar          float64
chlorides               float64
free_sulfur_dioxide     float64
total_sulfur_dioxide    float64
density                 float64
ph                      float64
sulphates               float64
alcohol                 float64
quality                   int64
wine_type                   str
dtype: object


,count,mean,std,min,25%,50%,75%,max
fixed_acidity,5320.0,7.215,1.320,3.800,6.400,7.000,7.700,15.900
volatile_acidity,5320.0,0.344,0.168,0.080,0.230,0.300,0.410,1.580
citric_acid,5320.0,0.318,0.147,0.000,0.240,0.310,0.400,1.660
residual_sugar,5320.0,5.048,4.500,0.600,1.800,2.700,7.500,65.800
chlorides,5320.0,0.057,0.037,0.009,0.038,0.047,0.066,0.611
free_sulfur_dioxide,5320.0,30.037,17.805,1.000,16.000,28.000,41.000,289.000
total_sulfur_dioxide,5320.0,114.109,56.774,6.000,74.000,116.000,153.250,440.000
density,5320.0,0.995,0.003,0.987,0.992,0.995,0.997,1.039
ph,5320.0,3.225,0.160,2.720,3.110,3.210,3.330,4.010
sulphates,5320.0,0.533,0.150,0.220,0.430,0.510,0.600,2.000


In [27]:
# Grouped view: quality by wine type
group_summary(df, group_col='wine_type', value_col='quality')

,count,mean,std,min,max
wine_type,,,,,
red,1359,5.623,0.824,3,8
white,3961,5.855,0.891,3,9


In [28]:
# Grouped view: average alcohol content by quality score
group_summary(df, group_col='quality', value_col='alcohol')

,count,mean,std,min,max
quality,,,,,
3,30,10.215,1.106,8.0,12.6
4,206,10.215,0.991,8.4,13.5
5,1752,9.872,0.828,8.0,14.9
6,2323,10.649,1.112,8.4,14.0
7,856,11.511,1.116,8.6,14.2
8,148,11.912,1.078,8.5,14.0
9,5,12.180,1.013,10.4,12.9


In [29]:
# Correlation check: which features relate most to quality?
correlations_with(df, target='quality')

alcohol                 0.469
citric_acid             0.098
free_sulfur_dioxide     0.054
sulphates               0.042
ph                      0.040
total_sulfur_dioxide   -0.050
residual_sugar         -0.057
fixed_acidity          -0.080
chlorides              -0.202
volatile_acidity       -0.265
density                -0.326
Name: quality, dtype: float64

**EDA observations.** Across the cleaned dataset, `alcohol` shows the strongest positive correlation with quality, while `volatile_acidity` and `density` are the most negatively associated. Grouped views show white and red wines have broadly similar average quality, and average alcohol content tends to rise for the higher-rated wines. These patterns worth visualizing next.